<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/06-complex-nested-data/01-structs.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 6.1 — Structs

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, IntegerType,
    DoubleType, DateType, ArrayType,
)

spark = (
    SparkSession.builder
    .appName("6.1-structs")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

EVENTS_SCHEMA = StructType([
    StructField("event_id", LongType(), False),
    StructField("ts",       StringType(), True),
    StructField("action",   StringType(), True),
    StructField("user", StructType([
        StructField("id",      StringType(), True),
        StructField("country", StringType(), True),
    ]), True),
    StructField("items", ArrayType(StructType([
        StructField("sku", StringType(),  True),
        StructField("qty", IntegerType(), True),
    ])), True),
    StructField("payment", StructType([
        StructField("method", StringType(), True),
        StructField("amount", DoubleType(), True),
    ]), True),
])
events = (
    spark.read.schema(EVENTS_SCHEMA)
    .option("mode", "DROPMALFORMED")
    .json(f"{DATA_DIR}/events.jsonl")
)

ORDERS_SCHEMA = StructType([
    StructField("order_id",    IntegerType(), False),
    StructField("customer_id", StringType(),  False),
    StructField("order_date",  DateType(),    True),
    StructField("product",     StringType(),  True),
    StructField("category",    StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("unit_price",  DoubleType(),  True),
    StructField("country",     StringType(),  True),
])
orders = (
    spark.read.option("header", True).schema(ORDERS_SCHEMA)
    .csv(f"{DATA_DIR}/orders.csv")
)

## Reading nested fields — dot paths, splat, graceful nulls

In [ ]:
events.select(
    "event_id",
    col("user.id").alias("user_id"),        # alias: leaf names collide otherwise
    "user.country",
    "payment.amount",                        # null struct -> null field, NO error
).show()

In [ ]:
events.select("event_id", "user.*").show(3)   # splat: fields hoisted one level

## Building structs — the argmax trick (3.4's unsolved exercise, solved)

In [ ]:
# per category: max unit_price AND the product achieving it — one aggregation
orders.groupBy("category").agg(
    F.max(F.struct("unit_price", "product")).alias("top")
).select("category", "top.product", "top.unit_price") \
 .orderBy("category").show(truncate=False)

In [ ]:
# Field order matters: struct comparison is field-by-field, left to right.
# struct(product, unit_price) would find the alphabetically-last product instead:
orders.groupBy("category").agg(
    F.max(F.struct("product", "unit_price")).alias("wrong")
).select("category", "wrong.product").orderBy("category").show(truncate=False)

## withField / dropFields — and the null-struct caveat

In [ ]:
patched = events.withColumn(
    "user",
    col("user").withField("country", F.coalesce(col("user.country"), F.lit("??"))),
)
patched.select("event_id", "user.*").show()

In [ ]:
# withField on a NULL struct does nothing (stays null):
attempt = events.withColumn("payment", col("payment").withField("method", F.lit("unknown")))
print("still-null payments:", attempt.filter(col("payment").isNull()).count())

# the fix: coalesce the STRUCT first, then the field exists
fixed = events.withColumn(
    "payment",
    F.coalesce(col("payment"), F.struct(F.lit("none").alias("method"), F.lit(0.0).alias("amount"))),
)
print("still-null payments:", fixed.filter(col("payment").isNull()).count())
fixed.select("event_id", "payment.*").show(4)

## Structs as travel bags: audit metadata as one column

In [ ]:
with_meta = orders.withColumn("_meta", F.struct(
    F.lit("orders.csv").alias("source_file"),
    F.current_timestamp().alias("ingested_at"),
))
with_meta.select("order_id", "_meta").show(3, truncate=False)
print("...and drops as one unit:", "_meta" not in with_meta.drop("_meta").columns)

## Exercises

1. Argmax, events edition: per `action`, the event_id with the highest `payment.amount` (watch what null payments do to max — is it correct here?).
2. Rename a nested field WITHOUT withField: rebuild `user` so `id` becomes `user_id`, using `F.struct` + aliases. Then do it again with `withField`+`dropFields`. Which reads better?
3. Nested-path withField: add `payment.currency = 'EUR'` for non-null payments only, in one `withColumn`.
4. Verify field-level column pruning: write `events` to parquet, read back selecting only `user.country`, and check `.explain()`'s ReadSchema — did it read the `items` leaf?

In [ ]:
# your exercise code here
